# 🖥️ Actividad 00a — Colab como servidor Ollama con GPU
## Tunneling con Ngrok · API REST · Clientes Python, curl y LangChain

---

En esta actividad convertirás Google Colab en un **servidor de LLMs accesible desde cualquier lugar**.
Una vez levantado, cualquier programa en tu computadora (o en otra notebook de Colab)
puede enviarle prompts y recibir respuestas — usando la GPU T4 de Google de forma gratuita.

```
Tu laptop / otra Colab
        │
        │  HTTP POST  (URL pública de Ngrok)
        ▼
  Ngrok Tunnel  ──►  Colab T4 GPU
                           │
                      Ollama Server
                      localhost:11434
                           │
                       TinyLlama
```

**⏱ Duración:** 30–40 minutos | **🎯 Resultado:** URL pública que sirve un LLM vía HTTP

---
## PARTE 1 · Verificar la GPU T4
**⏱ 2 minutos**

Antes de cualquier cosa, confirma que Colab te asignó GPU.
Ve a **Entorno de ejecución → Cambiar tipo → T4 GPU** si aún no lo hiciste.

In [1]:
import subprocess

# nvidia-smi muestra el estado de la GPU: modelo, memoria usada, temperatura
resultado = subprocess.run(['nvidia-smi'], capture_output=True, text=True)

if resultado.returncode == 0:
    print(resultado.stdout)
    print('✅ GPU detectada y lista')
else:
    print('❌ No hay GPU disponible.')
    print('   Ve a Entorno de ejecución → Cambiar tipo de entorno → T4 GPU')

Sun May 31 13:35:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import psutil

# Verificar RAM del sistema disponible para modelos
ram = psutil.virtual_memory()
print(f'RAM total   : {ram.total / 1e9:.1f} GB')
print(f'RAM libre   : {ram.available / 1e9:.1f} GB')
print(f'RAM usada   : {ram.percent:.1f}%')
print('✅ Recursos verificados')

RAM total   : 13.6 GB
RAM libre   : 12.5 GB
RAM usada   : 8.1%
✅ Recursos verificados


---
## PARTE 2 · Instalar Ollama
**⏱ 3 minutos**

Ollama es un servidor que descarga y gestiona modelos LLM localmente.
Lo instala un script oficial con `curl` directamente en el sistema de Colab.

In [4]:
# Instalar dependencia y Ollama
!apt-get install -y zstd -q
!curl -fsSL https://ollama.com/install.sh | sh

# Verificar instalación
import shutil
if shutil.which("ollama"):
    print("✅ Ollama instalado correctamente")
else:
    print("❌ Error: Ollama no se instaló correctamente")

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 51 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (15.6 MB/s)
Selecting previously unselected package zstd.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user 

In [5]:
# Instalar librerías Python que usaremos para probar el servidor
!pip install -q requests pyngrok langchain-ollama langchain openai
print('✅ Dependencias Python instaladas')

✅ Dependencias Python instaladas


---
## PARTE 3 · Arrancar el servidor y descargar el modelo
**⏱ 5 minutos**

Ollama funciona como un proceso servidor en background.
Lo arrancamos con `subprocess.Popen` para que no bloquee el notebook,
esperamos a que esté listo y luego descargamos el modelo.

In [6]:
import subprocess
import time
import requests

# subprocess.Popen lanza el proceso en paralelo (no bloquea el notebook)
# DEVNULL descarta la salida del servidor para no saturar la consola
ollama_process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print('Arrancando servidor Ollama...', end='')

# Esperar hasta que el servidor responda en el puerto 11434
for intento in range(30):
    try:
        resp = requests.get('http://localhost:11434', timeout=2)
        if resp.status_code == 200:
            print(f' listo en {intento+1}s')
            break
    except requests.exceptions.ConnectionError:
        print('.', end='', flush=True)
        time.sleep(1)
else:
    print('\n❌ El servidor no arrancó. Reinicia el runtime y vuelve a ejecutar.')

print('✅ Servidor Ollama activo en localhost:11434')

Arrancando servidor Ollama.... listo en 2s
✅ Servidor Ollama activo en localhost:11434


In [7]:
# 🔧 PARÁMETRO: cambia el modelo si quieres uno diferente
# Opciones ligeras para T4: tinyllama (~637 MB), qwen2:0.5b (~352 MB), phi3:mini (~2.2 GB)
MODELO = 'tinyllama'

print(f'Descargando modelo {MODELO}... (puede tardar 2-5 min la primera vez)')
resultado = subprocess.run(
    ['ollama', 'pull', MODELO],
    capture_output=True, text=True
)

if resultado.returncode == 0:
    print(f'✅ Modelo {MODELO} listo')
else:
    print(f'❌ Error: {resultado.stderr}')

Descargando modelo tinyllama... (puede tardar 2-5 min la primera vez)
✅ Modelo tinyllama listo


---
## PARTE 4 · Probar el servidor localmente
**⏱ 5 minutos**

Antes de exponerlo al exterior, verificamos que el servidor responde correctamente
con tres tipos de clientes: `requests` de Python, la API de generate, y la API compatible con OpenAI.

In [8]:
import json

# Probar la API /api/generate (API nativa de Ollama)
print('=== TEST: API /api/generate ===')
resp = requests.post(
    'http://localhost:11434/api/generate',
    json={
        'model': MODELO,       # Modelo a usar
        'prompt': '¿Qué es un LLM? En una sola frase.',
        'stream': False        # False = espera la respuesta completa antes de devolver
    }
)

if resp.status_code == 200:
    respuesta = resp.json()['response']
    print(f'Respuesta: {respuesta}')
    print('✅ API /api/generate funciona')
else:
    print(f'❌ Error {resp.status_code}: {resp.text}')

=== TEST: API /api/generate ===
Respuesta: Un LLM (Latin Linguistic Markup) es un sistema de anotación para la edición, clasificación y indexación del lenguaje natural en la información digital. Se trata de una tecnología que ayuda a los códigos fuertes a identificar y clasificar los datos de texto, además de hacer referencia a las bibliotecas de textos y otros recursos digitales. El LLM se basa en una serie de tecnologías para la información y se integra con el software en tiempo real.

La creación de un LLM requiere una búsqueda intensiva de patrones sintácticos, gramaticales y lexíco-semánticos que sean reconocidos por una red global de reconocedoras de idiomas y de linguistas especializados en lenguas. Los resultados de la anotación son una clasificación de los datos en base a su estructura (cantidad de palabras, posiciones en el texto) y la información que permite identificar cada palabra y entender cómo se usa (estilo, tono, temas, etc.).

Por ejemplo, un LLM puede ayudar a ident

In [9]:
from openai import OpenAI

# Ollama también expone una API compatible con OpenAI en /v1
# Esto permite usar el SDK oficial de OpenAI apuntando a Ollama
client_local = OpenAI(
    base_url='http://localhost:11434/v1',  # Ollama en local
    api_key='ollama'                        # Ollama no valida el key, pero lo requiere el SDK
)

print('=== TEST: API compatible OpenAI ===')
resp_openai = client_local.chat.completions.create(
    model=MODELO,
    messages=[{'role': 'user', 'content': '¿Cuál es la capital de Francia? Solo el nombre.'}]
)

print(f'Respuesta: {resp_openai.choices[0].message.content}')
print('✅ API compatible con OpenAI funciona')

=== TEST: API compatible OpenAI ===
Respuesta: No hay una capitalidad (capital) en Francia, solo el nombre.
✅ API compatible con OpenAI funciona


In [10]:
# Ver qué modelos están disponibles en este servidor
modelos_disponibles = requests.get('http://localhost:11434/api/tags').json()
print('=== MODELOS DISPONIBLES EN EL SERVIDOR ===')
for m in modelos_disponibles.get('models', []):
    size_gb = m.get('size', 0) / 1e9
    print(f"  {m['name']:<30} {size_gb:.2f} GB")
print('✅ Servidor funcionando correctamente en local')

=== MODELOS DISPONIBLES EN EL SERVIDOR ===
  tinyllama:latest               0.64 GB
✅ Servidor funcionando correctamente en local


---
## PARTE 5 · Exponer el servidor al exterior con Ngrok
**⏱ 5 minutos**

Hasta aquí el servidor solo es accesible desde dentro de Colab.
**Ngrok** crea un túnel seguro que asigna una URL pública (`https://xxxx.ngrok-free.app`)
que redirige el tráfico hacia `localhost:11434`.

```
Internet ──► https://xxxx.ngrok-free.app ──► localhost:11434 (Ollama)
```

> ⚠️ **Seguridad:** cualquiera con la URL puede usar tu GPU. No compartas la URL en foros públicos.
> El túnel se cierra cuando el runtime de Colab se desconecta.

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

# Leer token desde Colab Secrets (más seguro que pegarlo en el código)
try:
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
    print('✅ Token leído desde Colab Secrets')
except Exception:
    # Fallback: pegar el token manualmente (solo si no tienes Secrets configurado)
    NGROK_TOKEN = 'PEGA_TU_TOKEN_AQUÍ'
    print('⚠️  Usando token manual. Configura Colab Secrets para mayor seguridad.')

# Autenticar con Ngrok
ngrok.set_auth_token(NGROK_TOKEN)
print('✅ Ngrok autenticado')

In [ ]:
# Cerrar túneles previos para evitar conflictos
ngrok.kill()

# Crear túnel HTTP apuntando al puerto de Ollama
tunnel = ngrok.connect(11434, 'http')
URL_PUBLICA = tunnel.public_url

print('=' * 60)
print(f'🌐 SERVIDOR OLLAMA DISPONIBLE EN:')
print(f'   {URL_PUBLICA}')
print('=' * 60)
print(f'\nUsa esta URL en tus clientes externos:')
print(f'  API generate : {URL_PUBLICA}/api/generate')
print(f'  API OpenAI   : {URL_PUBLICA}/v1/chat/completions')
print(f'  Modelos      : {URL_PUBLICA}/api/tags')
print('\n⚠️  La URL cambia cada vez que reinicias el tunnel. Compártela con la clase ahora.')

---
## PARTE 6 · Conectarse al servidor desde clientes externos
**⏱ 10 minutos**

Ahora probamos el servidor usando la URL pública, como si fuera un API remoto.
Estas celdas representan lo que ejecutaría **otro estudiante** en su máquina o notebook.

In [ ]:
# === CLIENTE 1: requests puro (API nativa Ollama) ===
# Este código funciona desde cualquier Python en cualquier máquina
print('=== CLIENTE requests → API /api/generate ===')

resp_ext = requests.post(
    f'{URL_PUBLICA}/api/generate',   # URL pública de Ngrok
    json={
        'model': MODELO,
        'prompt': 'Explica qué es RAG en dos oraciones.',
        'stream': False
    },
    headers={'ngrok-skip-browser-warning': 'true'}  # Evitar la página de aviso de Ngrok
)

print(f'Status: {resp_ext.status_code}')
print(f'Respuesta: {resp_ext.json()["response"]}')
print('✅ Cliente requests funciona via Ngrok')

In [ ]:
# === CLIENTE 2: SDK oficial de OpenAI apuntando al servidor Ollama ===
# Cualquier app que use openai.ChatCompletion puede redirigirse a este servidor
print('=== CLIENTE OpenAI SDK → /v1/chat/completions ===')

client_ext = OpenAI(
    base_url=f'{URL_PUBLICA}/v1',    # URL pública en lugar de api.openai.com
    api_key='ollama',                 # Ollama acepta cualquier key
    default_headers={'ngrok-skip-browser-warning': 'true'}
)

resp_sdk = client_ext.chat.completions.create(
    model=MODELO,
    messages=[
        {'role': 'system', 'content': 'Eres un asistente de IA conciso.'},
        {'role': 'user',   'content': '¿Qué es el aprendizaje supervisado?'}
    ]
)
print(f'Respuesta: {resp_sdk.choices[0].message.content}')
print('✅ Cliente OpenAI SDK funciona via Ngrok')

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# === CLIENTE 3: LangChain apuntando al servidor remoto ===
# base_url permite redirigir LangChain a cualquier servidor Ollama remoto
print('=== CLIENTE LangChain → servidor remoto ===')

llm_remoto = ChatOllama(
    model=MODELO,
    base_url=URL_PUBLICA,            # URL pública del servidor
    client_kwargs={'headers': {'ngrok-skip-browser-warning': 'true'}}
)

respuesta_lc = llm_remoto.invoke([HumanMessage(content='¿Qué es LangChain? En una oración.')])
print(f'Respuesta: {respuesta_lc.content}')
print('✅ LangChain conectado al servidor remoto')

In [ ]:
# === CLIENTE 4: curl desde terminal ===
# Muestra el comando exacto para usar desde cualquier terminal
print('Para usar desde tu terminal (Mac/Linux/Windows):')
print()
print('# API nativa Ollama:')
print(f'curl -s {URL_PUBLICA}/api/generate \\')
print('  -H "ngrok-skip-browser-warning: true" \\')
print('  -d \'{"model": "' + MODELO + '", "prompt": "Hola", "stream": false}\'')
print()
print('# API compatible OpenAI:')
print(f'curl -s {URL_PUBLICA}/v1/chat/completions \\')
print('  -H "Content-Type: application/json" \\')
print('  -H "ngrok-skip-browser-warning: true" \\')
print('  -d \'{"model": "' + MODELO + '", "messages": [{"role": "user", "content": "Hola"}]}\'')

---
## PARTE 7 · Mantener el servidor activo
**⏱ 3 minutos**

Colab desconecta el runtime si detecta inactividad (~90 min sin ejecutar celdas).
Esta celda muestra cómo verificar que el servidor sigue activo y cómo regenerar la URL si el túnel cae.

In [ ]:
# Verificar estado del servidor y del túnel
def verificar_servidor():
    try:
        # Comprobar Ollama
        resp = requests.get('http://localhost:11434', timeout=3)
        print(f'Ollama local  : ✅ OK (status {resp.status_code})')
    except Exception:
        print('Ollama local  : ❌ No responde — reinicia desde PARTE 3')

    try:
        # Comprobar túnel Ngrok
        tunnels = ngrok.get_tunnels()
        if tunnels:
            print(f'Túnel Ngrok   : ✅ Activo → {tunnels[0].public_url}')
        else:
            print('Túnel Ngrok   : ❌ No hay túnel activo — re-ejecuta PARTE 5')
    except Exception as e:
        print(f'Túnel Ngrok   : ❌ Error — {e}')

verificar_servidor()

---
## 💬 Preguntas de reflexión

> **1. ¿Cuál es la diferencia entre la API `/api/generate` de Ollama y la API `/v1/chat/completions` compatible con OpenAI?**
> ¿Cuándo usarías cada una?
>
> *(Escribe aquí)*

> **2. El header `ngrok-skip-browser-warning: true` es necesario en todos los clientes. ¿Por qué crees que Ngrok añade esa pantalla de advertencia por defecto?**
>
> *(Escribe aquí)*

> **3. ¿Qué pasaría si dos estudiantes levantan su propio servidor y comparten sus URLs entre sí? ¿Podrían usar el modelo del otro?**
>
> *(Escribe aquí)*

---

## ✅ Resumen

| Lo que levantaste | URL |
|---|---|
| API nativa Ollama | `TU_URL/api/generate` |
| API compatible OpenAI | `TU_URL/v1/chat/completions` |
| Lista de modelos | `TU_URL/api/tags` |

**Guarda tu URL** — la necesitarás en las demás actividades del curso.

---
## 🧹 Limpieza (ejecutar al terminar la sesión)

In [ ]:
# Cerrar en orden: primero el túnel, luego el servidor
ngrok.kill()                        # Cierra el túnel Ngrok
ollama_process.terminate()          # Detiene el proceso Ollama
print('✅ Servidor y túnel cerrados correctamente')